# Signal-to-noise ratio figure

Generates the SNR interval figure used to illustrate the window-level measurement procedure.


In [ ]:
# ==========================================
# 1. IMPORTS AND PATHS
# ==========================================
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


def _find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "data" / "raw").is_dir() and (p / "src").is_dir():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
# Set REPO_ROOT explicitly if automatic detection fails.
# REPO_ROOT = Path(r"M:\Github\TFM_SeismicPhasePicker_ColombianAndesRegion")

sys.path.insert(0, str(REPO_ROOT / "src" / "preprocessing"))
import preprocessing_utils as pp  # noqa: E402

WAVEFORMS_DIR = REPO_ROOT / "data" / "raw" / "waveforms"
LABELING_DIR  = REPO_ROOT / "data" / "processed" / "labeling"
FIGURES_DIR   = REPO_ROOT / "data" / "processed" / "images" / "snr"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Figuras ->", FIGURES_DIR)

In [ ]:
# ==========================================
# 2. CONFIGURATION
# ==========================================
STATION   = "ZAR"     # station shown
SNR_INDEX = 0         # candidate row index

SNR_WIN_S = 5.0       # length of each measurement interval
CMP_WIN_S = 1.0       # horizontal-component comparison interval
SP_MIN, SP_MAX = 12, 20   # P-S separation range, seconds

# Figure style
PROC_COLOR   = "crimson"
P_COLOR      = "k"
S_COLOR      = "#1f77b4"
NOISE_COLOR  = "#8c8c8c"
SIGNAL_COLOR = "#1f4e9c"

DPI = 400                      # export resolution
SAVE_FORMATS = ["pdf", "png"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"]  = 42

sta_dir = WAVEFORMS_DIR / STATION
fs = pp.SAMPLING_RATE_HZ


def save_fig(fig, name):
    for fmt in SAVE_FORMATS:
        out = FIGURES_DIR / f"{name}.{fmt}"
        fig.savefig(out, dpi=DPI, bbox_inches="tight")
        print("guardado", out)

In [ ]:
# ==========================================
# 3. CANDIDATE WINDOWS
# ==========================================
# Select windows with four-model agreement on both phases and P-S
# separation within the configured range.
lab = pd.read_csv(LABELING_DIR / f"labels_{STATION}.csv")
lab = lab[(lab.category == "earthquake")
          & lab.p_sample.notna() & lab.s_sample.notna()].copy()
lab["sp_s"] = (lab.s_sample - lab.p_sample) / fs

sel = (lab[lab.sp_s.between(SP_MIN, SP_MAX)
           & (lab.p_n_models == 4) & (lab.s_n_models == 4)]
       .sort_values("snr_p_db", ascending=False)
       .reset_index(drop=True))

print(f"{len(sel)} candidatas en {STATION}. Diez mejores:\n")
print(sel[["timestamp", "sp_s", "snr_p_db", "snr_s_db"]]
      .head(10).to_string(float_format=lambda v: f"{v:,.2f}"))

In [ ]:
# ==========================================
# 4. FIGURE
# ==========================================
r  = sel.iloc[SNR_INDEX]
ts = r.timestamp
ps, ss = int(r.p_sample), int(r.s_sample)
p_s, s_s = ps / fs, ss / fs

arr = pp.load_window_array(sta_dir, ts, preprocess=True)
if arr is None:
    raise RuntimeError(f"{ts}: la ventana no superó el control de calidad.")

t    = np.arange(pp.NPTS) / fs
ncmp = int(CMP_WIN_S * fs)
amps = [np.abs(arr[i, ss - ncmp:ss + ncmp]).max() for i in (1, 2)]
h    = 1 + int(np.argmax(amps))        # horizontal empleada para la onda S

fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

for i, comp in enumerate("ZNE"):
    ax[i].plot(t, arr[i], lw=0.7, color=PROC_COLOR, rasterized=True)
    ax[i].axvline(p_s, color=P_COLOR, ls="--", lw=1.2,
                  label="Onda P" if i == 0 else None)
    ax[i].axvline(s_s, color=S_COLOR, ls="--", lw=1.2,
                  label="Onda S" if i == 0 else None)

    if i == 0:                                   # la onda P se mide en la vertical
        ax[i].axvspan(p_s - SNR_WIN_S, p_s, color=NOISE_COLOR,  alpha=0.35)
        ax[i].axvspan(p_s, p_s + SNR_WIN_S, color=SIGNAL_COLOR, alpha=0.22)
        nota = "Intervalos de la onda P"
    elif i == h:                                 # la onda S, en la horizontal elegida
        ax[i].axvspan(s_s - SNR_WIN_S, s_s, color=NOISE_COLOR,  alpha=0.35)
        ax[i].axvspan(s_s, s_s + SNR_WIN_S, color=SIGNAL_COLOR, alpha=0.22)
        nota = "Intervalos de la onda S"
    else:                                        # horizontal descartada
        nota = "Componente no empleada"

    if i in (1, 2):                              # entorno de comparación
        ax[i].axvspan(s_s - CMP_WIN_S, s_s + CMP_WIN_S, facecolor="none",
                      edgecolor="0.35", ls=":", lw=0.8)

    ax[i].text(0.012, 0.92, nota, transform=ax[i].transAxes, fontsize=8, va="top",
               bbox=dict(boxstyle="round,pad=0.25", fc="white",
                         ec="0.7", lw=0.5, alpha=0.85))
    ax[i].set_ylabel(f"{comp}\nAmplitud")
    ax[i].margins(x=0)

ax[0].set_xlim(max(0, p_s - SNR_WIN_S - 5), min(t[-1], s_s + SNR_WIN_S + 15))
ax[0].legend(handles=[*ax[0].get_legend_handles_labels()[0],
                      Patch(facecolor=NOISE_COLOR,  alpha=0.35, label="Ruido, 5 s"),
                      Patch(facecolor=SIGNAL_COLOR, alpha=0.22, label="Señal, 5 s")],
             loc="upper right", framealpha=0.9, fontsize=8, ncol=2)
ax[-1].set_xlabel("Tiempo (s)")
fig.suptitle(f"Estación: {STATION}\nRegistro: {ts}")
plt.tight_layout()

print(f"[{ts}]  separación S-P = {r.sp_s:.2f} s")
print(f"  onda P en Z  -> SNR = {r.snr_p_db:.1f} dB")
print(f"  onda S en {'NE'[h-1]}  -> SNR = {r.snr_s_db:.1f} dB"
      f"   (amplitudes N={amps[0]:.0f}, E={amps[1]:.0f})")

plt.show()

In [ ]:
# ==========================================
# 5. SAVE
# ==========================================
# Save after selecting the illustrative window.
save_fig(fig, f"IntervalosSNR_{STATION}_{ts}")